# Ablation comparison

Read-only until every validation passes. This notebook reads the six variant `study.json`, `summary.json`, and `candidate_status.csv` files, verifies that the shared study identity and expected ablation switches match, then writes `<STUDY_ROOT>/comparison.csv`.

In partial mode, it reports missing incident accounting as `n_unreported`. It reports candidate detections only, not accuracy, precision, recall, or IoU.

Different raw commit SHAs are accepted only after verifying identical Hub blob IDs for every selected raw input file (no imagery download). The CSV preserves each original SHA and records `raw_identity_check`; genuinely changed or unverifiable inputs block publication.

## Kaggle setup

1. In **Settings -> Internet**, turn internet **on**.
2. Add the Kaggle secret `huggingface_token` with read/write access to the output dataset repository.
3. Run this notebook after the run notebooks have saved their results.
4. `ALLOW_PARTIAL = True` is the default: incomplete runs produce a diagnostic report with warnings, failure/unreported counts, run status, and metadata completeness. All six matching study manifests are still required. Set `ALLOW_PARTIAL = False` to require complete runs.
5. Partial candidate totals are not a complete scientific comparison; do not treat failed or unreported incidents as zero detections.

In [ ]:
%pip install -q "huggingface_hub>=0.23"

In [ ]:
from __future__ import annotations

import csv
import io
import json
import os
import random
import re
import time
from pathlib import Path, PurePosixPath

from huggingface_hub import HfApi, hf_hub_download

STUDY_ROOT = "ablations/study_01"
OUTPUT_REPO = "sasudo2/landslides"
OUTPUT_REVISION = "main"
ALLOW_PARTIAL = True
COMPARISON_FILENAME = "comparison.csv"

STUDY_SCHEMA_VERSION = "ablation-study-v2"
OUTPUT_SCHEMA_VERSION = "ablation-output-v1"
DETECTION_ALGORITHM_VERSION = "ablation-detection-v2"
VARIANTS = {
    "baseline": {"modality": "fused", "slope": True, "slic": True, "bonferroni": True},
    "optical_only": {"modality": "optical", "slope": True, "slic": True, "bonferroni": True},
    "sar_only": {"modality": "sar", "slope": True, "slic": True, "bonferroni": True},
    "no_slope": {"modality": "fused", "slope": False, "slic": True, "bonferroni": True},
    "no_slic": {"modality": "fused", "slope": True, "slic": False, "bonferroni": True},
    "no_bonferroni": {"modality": "fused", "slope": True, "slic": True, "bonferroni": False},
}
SHARED_IDENTITY_KEYS = (
    "study_schema_version",
    "output_schema_version",
    "detection_algorithm_version",
    "raw_repo",
    "raw_commit_sha",
    "source_csv_name",
    "source_csv_sha256",
    "cohort",
    "incident_ids",
    "required_raw_inputs",
    "before_file_by_incident",
    "settings",
    "package_versions",
)
COMPARISON_FIELDS = [
    "variant",
    "cohort_size",
    "n_successful",
    "n_failed",
    "n_unreported",
    "n_incidents_with_candidates",
    "n_candidates",
    "candidates_per_successful_incident",
    "raw_commit_sha",
    "study_identity_check",
    "raw_identity_check",
    "run_complete",
    "run_status",
    "metadata_complete",
    "switches",
]
TRANSIENT_STATUS_CODES = {408, 409, 412, 425, 429, 500, 502, 503, 504}


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise RuntimeError("Set HF_TOKEN or run on Kaggle with the huggingface_token secret attached") from exc
    try:
        return UserSecretsClient().get_secret("huggingface_token")
    except Exception as exc:
        raise RuntimeError("Attach the Kaggle secret 'huggingface_token' with read access") from exc


HF_TOKEN = resolve_hf_token()
API = HfApi(token=HF_TOKEN)


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def _status_code(exc):
    response = getattr(exc, "response", None)
    return getattr(response, "status_code", None)


def is_transient_hub_error(exc):
    status = _status_code(exc)
    if status in TRANSIENT_STATUS_CODES:
        return True
    if isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    message = str(exc).lower()
    markers = (
        "timeout", "timed out", "temporarily", "connection reset", "connection aborted",
        "rate limit", "too many requests", "server error", "bad gateway",
        "service unavailable", "gateway timeout",
    )
    return any(marker in message for marker in markers) or status in TRANSIENT_STATUS_CODES


def retry_hub(action, description, attempts=5, base_delay=2.0):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:
            if attempt >= attempts or not is_transient_hub_error(exc):
                raise
            delay = min(base_delay ** attempt, 60.0) + random.uniform(0.0, 1.0)
            print(f"{description} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Retry loop exhausted for {description}")


def raw_input_blobs(study):
    repo = study["raw_repo"]
    sha = study["raw_commit_sha"]
    if not isinstance(sha, str) or re.fullmatch(r"[0-9a-fA-F]{40}", sha) is None:
        raise RuntimeError(f"Cannot verify raw inputs: {repo}@{sha} is not an immutable SHA")
    paths = []
    for incident_id in study["incident_ids"]:
        before = study["before_file_by_incident"].get(str(incident_id))
        if before not in ("planet_before.tif", "gee_before.tif"):
            raise RuntimeError(f"Cannot verify raw inputs: invalid before file for incident {incident_id}")
        prefix = f"raw_images/raw_incidents/incident_{incident_id}/incident_{incident_id}_"
        paths.extend(prefix + suffix for suffix in (
            before, "after.tif", "slope.tif", "sar_pre.tif", "sar_post.tif"
        ))
    blobs = {}
    for start in range(0, len(paths), 100):
        batch = paths[start:start + 100]
        entries = retry_hub(
            lambda batch=batch: API.get_paths_info(
                repo_id=repo, repo_type="dataset", revision=sha, paths=batch,
            ),
            f"verify raw input blobs at {repo}@{sha}",
        )
        for entry in entries:
            path = getattr(entry, "path", None)
            blob = getattr(entry, "blob_id", None)
            if path in batch and isinstance(blob, str) and blob:
                blobs[path] = blob
    missing = sorted(set(paths) - blobs.keys())
    if missing:
        raise RuntimeError(
            f"Cannot verify raw inputs at {repo}@{sha}: missing file/blob metadata for "
            f"{len(missing)} input(s), including {missing[:5]}"
        )
    return blobs


def repo_files():
    def list_tree():
        return list(API.list_repo_tree(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            path_in_repo=study_root,
            recursive=True,
        ))

    try:
        entries = retry_hub(list_tree, f"list files under {OUTPUT_REPO}@{OUTPUT_REVISION}/{study_root}")
    except Exception as exc:
        status = _status_code(exc)
        message = str(exc).lower()
        if status == 404 or "not found" in message or "entry not found" in message:
            return set()
        raise
    files = set()
    for entry in entries:
        path = getattr(entry, "path", None)
        if path and (getattr(entry, "type", None) == "file" or hasattr(entry, "blob_id")):
            files.add(path)
    return files


def download(path):
    return retry_hub(
        lambda: hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            filename=path,
            token=HF_TOKEN,
        ),
        f"download {path}",
    )


def read_json(path):
    return json.loads(Path(download(path)).read_text(encoding="utf-8"))


def read_csv(path):
    with Path(download(path)).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def derive_counts(study, summary, status_rows, name, allow_partial=False):
    ids = study["incident_ids"]
    errors = []
    if summary is not None:
        if summary.get("variant") != name:
            errors.append(f"{name}: summary variant is {summary.get('variant')!r}")
        try:
            counts = {
                "n_incidents": int(summary["n_incidents"]),
                "n_ok": int(summary["n_ok"]),
                "n_failed": int(summary["n_failed"]),
                "n_candidates": int(summary["n_candidates"]),
                "n_incidents_with_candidates": int(summary["n_incidents_with_candidates"]),
            }
        except (KeyError, TypeError, ValueError) as exc:
            errors.append(f"{name}: invalid summary.json ({exc})")
            counts = None
    else:
        counts = None
    status_ids = []
    if status_rows is not None:
        failed = 0
        ok = 0
        candidates = 0
        with_candidates = 0
        for row in status_rows:
            try:
                incident_id = int(row["incident_id"])
            except (KeyError, TypeError, ValueError) as exc:
                errors.append(f"{name}: invalid candidate_status.csv row ({exc})")
                continue
            status_ids.append(incident_id)
            status = row.get("status")
            if status not in {"ok", "failed"}:
                errors.append(f"{name}: invalid candidate status {status!r}")
                continue
            try:
                n = int(row["n_candidates"])
            except (KeyError, TypeError, ValueError) as exc:
                errors.append(f"{name}: invalid candidate count ({exc})")
                continue
            if n < 0:
                errors.append(f"{name}: negative candidate count {n}")
                continue
            if status == "ok":
                ok += 1
                candidates += n
                with_candidates += int(n > 0)
            else:
                failed += 1
        status_positions = [ids.index(incident_id) for incident_id in status_ids if incident_id in ids]
        if len(status_positions) != len(status_ids) or status_positions != sorted(set(status_positions)):
            errors.append(f"{name}: candidate_status.csv has unknown, duplicate, or unordered incident IDs")
        elif status_ids != ids and not allow_partial:
            errors.append(f"{name}: candidate_status.csv does not contain the study cohort in order")
        status_counts = {
            "n_incidents": len(status_rows),
            "n_ok": ok,
            "n_failed": failed,
            "n_candidates": candidates,
            "n_incidents_with_candidates": with_candidates,
        }
        if counts is None:
            counts = status_counts
        elif status_ids == ids and counts != status_counts:
            errors.append(f"{name}: summary.json and candidate_status.csv disagree")
    if counts is None:
        counts = {
            "n_incidents": 0,
            "n_ok": 0,
            "n_failed": 0,
            "n_candidates": 0,
            "n_incidents_with_candidates": 0,
        }
    if counts["n_incidents"] > len(ids):
        errors.append(f"{name}: incident count exceeds the study cohort")
    if counts["n_ok"] + counts["n_failed"] != counts["n_incidents"]:
        errors.append(f"{name}: successful and failed counts do not sum to incident count")
    if counts["n_incidents_with_candidates"] > counts["n_ok"]:
        errors.append(f"{name}: incidents with candidates exceeds successful incidents")
    if counts["n_candidates"] < counts["n_incidents_with_candidates"]:
        errors.append(f"{name}: candidate total is below the incidents-with-candidates count")
    counts_complete = (
        counts["n_incidents"] == len(ids)
        and counts["n_failed"] == 0
        and counts["n_ok"] == len(ids)
        and (summary is None or summary.get("metadata_complete", True))
    )
    if summary is not None and summary.get("complete") is not counts_complete:
        errors.append(f"{name}: summary complete flag disagrees with incident counts")
    complete = counts_complete and (status_rows is None or status_ids == ids)
    return counts, complete, errors


def validate_study_root(value):
    path = PurePosixPath(value)
    if (
        not path.parts
        or path.is_absolute()
        or value != path.as_posix()
        or any(part in (".", "..") or part.startswith(".") for part in path.parts)
        or path.parts[0] in ("candidates", "raw_images")
    ):
        raise ValueError("STUDY_ROOT must be a safe relative folder such as ablations/study_01")
    return value


study_root = validate_study_root(STUDY_ROOT.strip())

files = repo_files()
missing_folders = [
    name for name in VARIANTS
    if not any(path.startswith(f"{study_root}/{name}/") for path in files)
]
if missing_folders:
    raise FileNotFoundError(
        "Missing variant folders in " + f"{OUTPUT_REPO}@{OUTPUT_REVISION}: {missing_folders}"
    )

required_files = ("study.json", "run.json", "candidate_status.csv", "candidate_metadata.csv", "diagnostics.json", "summary.json")
studies = {}
summaries = {}
statuses = {}
runs = {}
problems = []
for name in VARIANTS:
    prefix = f"{study_root}/{name}/"
    present = {path[len(prefix):] for path in files if path.startswith(prefix)}
    study_path = prefix + "study.json"
    if "study.json" not in present:
        problems.append(f"{name}: missing study.json")
        continue
    study = read_json(study_path)
    if study.get("variant") != name:
        problems.append(f"{name}: study.json variant is {study.get('variant')!r}")
    if study.get("switches") != VARIANTS[name]:
        problems.append(f"{name}: unexpected ablation switches {study.get('switches')!r}")
    if study.get("study_schema_version") != STUDY_SCHEMA_VERSION:
        problems.append(f"{name}: unexpected study schema version")
    if study.get("output_schema_version") != OUTPUT_SCHEMA_VERSION:
        problems.append(f"{name}: unexpected output schema version")
    if study.get("detection_algorithm_version") != DETECTION_ALGORITHM_VERSION:
        problems.append(f"{name}: unexpected detection algorithm version")
    cohort = study.get("cohort", {})
    ids = study.get("incident_ids", [])
    if cohort.get("declaration") != "df.iloc[900:1000]" or cohort.get("start") != 900 or cohort.get("stop") != 1000:
        problems.append(f"{name}: unexpected cohort declaration {cohort!r}")
    if len(ids) != 100:
        problems.append(f"{name}: cohort has {len(ids)} incident IDs, expected 100")
    if len(set(ids)) != len(ids):
        problems.append(f"{name}: cohort contains duplicate incident IDs")
    studies[name] = study

    if ALLOW_PARTIAL:
        summary = read_json(prefix + "summary.json") if "summary.json" in present else None
        status_rows = read_csv(prefix + "candidate_status.csv") if "candidate_status.csv" in present else None
    else:
        absent = [filename for filename in required_files if filename not in present]
        if absent:
            problems.append(f"{name}: incomplete folder, missing {absent}")
        summary = read_json(prefix + "summary.json") if "summary.json" in present else None
        status_rows = read_csv(prefix + "candidate_status.csv") if "candidate_status.csv" in present else None
    run_record = read_json(prefix + "run.json") if "run.json" in present else None
    if not ALLOW_PARTIAL:
        if run_record is None:
            problems.append(f"{name}: missing run.json")
        elif run_record.get("status") != "complete":
            problems.append(f"{name}: run.json status is {run_record.get('status')!r}")
    summaries[name] = summary
    statuses[name] = status_rows
    runs[name] = run_record

if problems:
    raise RuntimeError(
        "Cannot compare runs yet: " + "; ".join(problems)
        + ". For incomplete-run diagnostics, set ALLOW_PARTIAL = True; "
        "study identity and accounting errors must still be corrected."
    )

if not studies:
    raise RuntimeError("No readable study manifests")

first_name = next(iter(VARIANTS))
reference_name = first_name if first_name in studies else next(iter(studies))
reference = studies[reference_name]
reference_ids = reference["incident_ids"]
conflicts = []
for name, study in studies.items():
    for key in SHARED_IDENTITY_KEYS:
        if key not in study:
            conflicts.append(f"{name}: missing shared field {key}")
        elif key != "raw_commit_sha" and canonical_json(study[key]) != canonical_json(reference[key]):
            conflicts.append(f"{name}: shared field {key} differs")
    if study.get("incident_ids") != reference_ids:
        conflicts.append(f"{name}: ordered cohort IDs differ")
if conflicts:
    raise RuntimeError("Cross-run identity check failed: " + "; ".join(conflicts))

raw_identity_checks = {name: "same_commit" for name in studies}
different_commits = {
    name: study for name, study in studies.items()
    if study["raw_commit_sha"] != reference["raw_commit_sha"]
}
if different_commits:
    reference_blobs = raw_input_blobs(reference)
    blobs_by_sha = {reference["raw_commit_sha"]: reference_blobs}
    for name, study in different_commits.items():
        sha = study["raw_commit_sha"]
        if sha not in blobs_by_sha:
            blobs_by_sha[sha] = raw_input_blobs(study)
        blobs = blobs_by_sha[sha]
        changed = sorted(path for path in reference_blobs if blobs.get(path) != reference_blobs[path])
        if changed:
            raise RuntimeError(
                f"Cross-run raw input check failed: {name}@{sha} differs from "
                f"{reference_name}@{reference['raw_commit_sha']} in {len(changed)} input(s), "
                f"including {changed[:5]}. Rerun affected variants with one RAW_COMMIT_SHA "
                "in a new STUDY_ROOT; do not rewrite existing manifests."
            )
        raw_identity_checks[name] = "identical_input_blobs"
        print(
            f"Verified {name}: raw commit {sha} differs from {reference_name}, "
            f"but all {len(reference_blobs)} selected input files have identical blob IDs."
        )

rows = []
completeness_errors = []
for name, study in studies.items():
    counts, complete, errors = derive_counts(
        study, summaries.get(name), statuses.get(name), name, allow_partial=ALLOW_PARTIAL
    )
    completeness_errors.extend(errors)
    run_status = runs[name].get("status", "unknown") if runs[name] is not None else "missing"
    complete = complete and run_status == "complete"
    if not complete and not ALLOW_PARTIAL:
        completeness_errors.append(f"{name}: run is not complete ({counts['n_failed']} failed incident(s))")
    rows.append({
        "variant": name,
        "cohort_size": len(study["incident_ids"]),
        "n_successful": counts["n_ok"],
        "n_failed": counts["n_failed"],
        "n_unreported": (
            max(0, len(study["incident_ids"]) - len(statuses[name]))
            if statuses[name] is not None
            else max(0, len(study["incident_ids"]) - counts["n_incidents"])
        ),
        "n_incidents_with_candidates": counts["n_incidents_with_candidates"],
        "n_candidates": counts["n_candidates"],
        "candidates_per_successful_incident": (counts["n_candidates"] / counts["n_ok"]) if counts["n_ok"] else 0.0,
        "raw_commit_sha": study["raw_commit_sha"],
        "study_identity_check": "pass",
        "raw_identity_check": raw_identity_checks[name],
        "run_complete": "yes" if complete else "no",
        "run_status": run_status,
        "metadata_complete": (
            summaries[name].get("metadata_complete", True)
            if summaries[name] is not None else "unknown"
        ),
        "switches": json.dumps(study["switches"], sort_keys=True),
    })

if completeness_errors:
    raise RuntimeError(
        "Cannot write comparison.csv: " + "; ".join(completeness_errors)
        + ". Set ALLOW_PARTIAL = True to write a partial diagnostic report."
    )

incomplete_rows = [row for row in rows if row["run_complete"] == "no"]
if incomplete_rows:
    print(
        "WARNING: PARTIAL DIAGNOSTIC REPORT. Some runs are incomplete; "
        "candidate totals are not a complete or necessarily comparable ablation result. "
        "Failed and unreported incidents are not zero detections, and missing metadata "
        "has not been reconstructed."
    )
    for row in incomplete_rows:
        print(
            f"WARNING: {row['variant']}: run_status={row['run_status']}, "
            f"successful={row['n_successful']}, failed={row['n_failed']}, "
            f"unreported={row['n_unreported']}, metadata_complete={row['metadata_complete']}"
        )

buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=COMPARISON_FIELDS)
writer.writeheader()
writer.writerows(rows)
comparison_bytes = buffer.getvalue().encode("utf-8")
comparison_path = f"{study_root}/{COMPARISON_FILENAME}"
retry_hub(
    lambda: API.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        revision=OUTPUT_REVISION,
        path_or_fileobj=io.BytesIO(comparison_bytes),
        path_in_repo=comparison_path,
        commit_message=f"Compare six ablations in {study_root}",
    ),
    f"upload {comparison_path}",
)

for row in rows:
    print(json.dumps(row, sort_keys=True))
print(f"Wrote {comparison_path}")